# $k$-nearest neighbors & local regression: ESL

*Based on Hastie, Tibshirani & Friedman (2009). The Elements of Statistical Learning. Chapter 2.*

TMDB movie data (`inputs/tmdb-movie-metadata/`): predict **revenue** from budget, popularity, votes, runtime, release year. Download with `uv run python scripts/download_tmdb_movie_metadata.py`. Same data as `least_squares_regression.ipynb`. Neighbor count $k$ vs train/test MSE, then linear vs $k$-NN on one input (e.g. `budget`).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import helpers

ROOT, INPUTS, OUTPUTS = helpers.init_paths()

In [2]:
X, y, target = helpers.load_tmdb_xy(INPUTS)
print(f"Loaded {len(X):,} rows, {X.shape[1]} numeric features, target={target!r}")

Loaded 3,229 rows, 6 numeric features, target='revenue'


## Bias–variance trade-off as a function of $k$

The $k$-NN prediction at $x_0$ is the average of the $k$ nearest training responses:

$$\hat{f}(x_0) = \frac{1}{k} \sum_{x_i \in \mathcal{N}_k(x_0)} y_i$$

- **Small $k$**: low bias (the model can hug the data), high variance (sensitive to individual points).
- **Large $k$**: high bias (over-smoothed), low variance.

Train MSE falls monotonically as $k$ decreases; test MSE has a minimum — the sweet spot between the two extremes.

In [3]:
fig, summary = helpers.knn_train_test_mse_figure(X, y, max_rows=40_000)
fig.show()

ks = summary["ks"]
print(
    f"Best k ({int(ks.min())}..{int(ks.max())}) by test MSE: {summary['k_best']}, "
    f"test MSE = {summary['min_test_mse']:.4f}"
)
print(f"Linear regression test MSE: {summary['linear_test_mse']:.4f}")

Best k (2..9) by test MSE: 9, test MSE = 9206583081710730.0000
Linear regression test MSE: 10219034266382514.0000


## Linear vs $k$-NN on a single feature

Projecting onto one feature (budget) makes the comparison visual. The linear fit is a global straight line; $k$-NN traces the local average of nearby training points. In sparse regions $k$-NN falls back toward the global mean, while the linear model extrapolates indefinitely — each approach has its failure mode.

In [4]:
fig2, _feat = helpers.linear_vs_knn_single_feature_figure(summary["X_train"], summary["y_train"], target)
fig2.show()

/Users/tomaskennedymartin/vsco code/github/financial-machine-learning/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/Users/tomaskennedymartin/vsco code/github/financial-machine-learning/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
